# HW2

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
from PIL import Image
import os
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
import itertools
import copy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

BASE_PATH = 'data/'
UNLABELED_PATH = os.path.join(BASE_PATH, 'train', 'unlabeled')
LABELED_PATH = os.path.join(BASE_PATH, 'train', 'labeled')
TEST_PATH = os.path.join(BASE_PATH, 'test')

PRETRAIN_EPOCHS = 100
FINETUNE_EPOCHS = 400
BATCH_SIZE = 128 
LEARNING_RATE_PRETRAIN = 1e-3
LEARNING_RATE_FINETUNE = 1e-4
GRID_SIZE = 3
PATCH_SIZE = 64
N_PERMUTATIONS = 100

device: cuda


## Rotations

In [18]:
class MultiTaskDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        self.to_grayscale = transforms.Grayscale(num_output_channels=3)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        
        is_color_label = 1
        if torch.rand(1) < 0.5:
            image = self.to_grayscale(image)
            is_color_label = 0 

        tensor_transforms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        image = tensor_transforms(image)
        
        rotation_label = torch.randint(0, 4, (1,)).item()
        rotated_image = torch.rot90(image, k=rotation_label, dims=[1, 2])

        return rotated_image, torch.tensor(rotation_label), torch.tensor(is_color_label)

pretext_pil_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
])

pretrain_dataset = MultiTaskDataset(UNLABELED_PATH, transform=pretext_pil_transforms)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Найдено {len(pretrain_dataset)} изображений для multi-task pre-training.")

class MultiTaskResNet(nn.Module):
    def __init__(self):
        super(MultiTaskResNet, self).__init__()
        base_model = models.resnet18()
        self.backbone = nn.Sequential(*list(base_model.children())[:-1])
        num_features = base_model.fc.in_features
        
        self.rotation_head = nn.Linear(num_features, 4)
        self.color_head = nn.Linear(num_features, 2)

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        
        rotation_out = self.rotation_head(features)
        color_out = self.color_head(features)
        
        return rotation_out, color_out

Найдено 16611 изображений для multi-task pre-training.


In [19]:
pretext_model = MultiTaskResNet().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pretext_model.parameters(), lr=LEARNING_RATE_PRETRAIN)
scheduler_pretrain = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)


print("Starting multi-task self-supervised pre-training...")

epoch_pbar = tqdm(range(PRETRAIN_EPOCHS), desc="Pre-training Progress")

for epoch in epoch_pbar:
    pretext_model.train()
    total_loss_epoch = 0.0
    rot_correct = 0
    color_correct = 0
    total_samples = 0

    batch_pbar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}", leave=False)

    for images, rot_labels, color_labels in batch_pbar:
        images = images.to(DEVICE)
        rot_labels = rot_labels.to(DEVICE)
        color_labels = color_labels.to(DEVICE)

        optimizer.zero_grad()

        rotation_out, color_out = pretext_model(images)
        
        loss_rot = criterion(rotation_out, rot_labels)
        loss_color = criterion(color_out, color_labels)
        
        total_loss = loss_rot + loss_color

        total_loss.backward()
        optimizer.step()

        total_loss_epoch += total_loss.item() * images.size(0)
        _, rot_predicted = torch.max(rotation_out.data, 1)
        _, color_predicted = torch.max(color_out.data, 1)
        
        total_samples += images.size(0)
        rot_correct += (rot_predicted == rot_labels).sum().item()
        color_correct += (color_predicted == color_labels).sum().item()
        
        batch_pbar.set_postfix(loss=total_loss.item())

    scheduler_pretrain.step()
    
    avg_loss = total_loss_epoch / total_samples
    rot_acc = rot_correct / total_samples
    color_acc = color_correct / total_samples
    
    epoch_pbar.set_postfix(Loss=f"{avg_loss:.4f}", Rot_Acc=f"{rot_acc:.2f}", Color_Acc=f"{color_acc:.2f}")

print("\nFinished pre-training.")

torch.save(pretext_model.backbone.state_dict(), 'resnet18_backbone_multitask_128.pth')
print("Saved pre-trained multitask backbone weights to 'resnet18_backbone_multitask_128.pth'")

Starting multi-task self-supervised pre-training...


Pre-training Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/100:   0%|          | 0/130 [00:00<?, ?it/s]

Epoch 2/100:   0%|          | 0/130 [00:00<?, ?it/s]

Epoch 3/100:   0%|          | 0/130 [00:00<?, ?it/s]

...

Epoch 97/100:   0%|          | 0/130 [00:00<?, ?it/s]

Epoch 98/100:   0%|          | 0/130 [00:00<?, ?it/s]

Epoch 99/100:   0%|          | 0/130 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/130 [00:00<?, ?it/s]


Finished pre-training.
Saved pre-trained multitask backbone weights to 'resnet18_backbone_multitask_128.pth'


In [6]:
def cutmix_data(x, y, alpha=1.0, device='cuda'):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    y_a, y_b = y, y[index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, y_a, y_b, lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def cutmix_criterion(criterion, outputs, y_a, y_b, lam):
    return lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)

In [7]:
finetune_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_labeled_dataset = datasets.ImageFolder(LABELED_PATH, transform=finetune_transforms)
train_size = int(0.9 * len(full_labeled_dataset))
val_size = len(full_labeled_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
class_names = full_labeled_dataset.classes

finetune_model = models.resnet18(num_classes=10)
print("Loading pre-trained multitask backbone weights...")
backbone_to_load = nn.Sequential(*list(models.resnet18().children())[:-1])
backbone_to_load.load_state_dict(torch.load('resnet18_backbone_multitask.pth'))
finetune_model.load_state_dict(backbone_to_load.state_dict(), strict=False)
finetune_model = finetune_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(finetune_model.parameters(), lr=LEARNING_RATE_FINETUNE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

best_val_acc = 0.0

print("\n--- Starting Stage 1: Training the classifier head ---")
for param in finetune_model.parameters():
    param.requires_grad = False
for param in finetune_model.fc.parameters():
    param.requires_grad = True

head_optimizer = optim.AdamW(finetune_model.fc.parameters(), lr=LEARNING_RATE_PRETRAIN) # Используем более высокий LR
FREEZE_EPOCHS = 10

for epoch in range(FREEZE_EPOCHS):
    finetune_model.train()
    for images, labels in tqdm(train_loader, desc=f"Head Training {epoch+1}/{FREEZE_EPOCHS}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        head_optimizer.zero_grad()
        outputs = finetune_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        head_optimizer.step()
    print(f"Head Training Epoch {epoch+1}/{FREEZE_EPOCHS} completed.")

Loading pre-trained multitask backbone weights...

--- Starting Stage 1: Training the classifier head ---


Head Training 1/10:   0%|          | 0/15 [00:00<?, ?it/s]

Head Training Epoch 1/10 completed.


Head Training 2/10:   0%|          | 0/15 [00:00<?, ?it/s]

Head Training Epoch 2/10 completed.


...

Head Training Epoch 8/10 completed.


Head Training 9/10:   0%|          | 0/15 [00:00<?, ?it/s]

Head Training Epoch 9/10 completed.


Head Training 10/10:   0%|          | 0/15 [00:00<?, ?it/s]

Head Training Epoch 10/10 completed.


In [8]:
finetune_model = models.resnet18(num_classes=10)

print("Loading pre-trained backbone weights...")
finetune_model.load_state_dict(torch.load('resnet18_backbone_pretrained.pth'), strict=False)
finetune_model = finetune_model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(finetune_model.parameters(), lr=LEARNING_RATE_FINETUNE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(finetune_model.state_dict())

print("Starting supervised fine-tuning with CutMix...")

epoch_pbar = tqdm(range(FINETUNE_EPOCHS), desc="Fine-tuning Progress")

for epoch in epoch_pbar:
    finetune_model.train()
    correct_predictions = 0
    total_predictions = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} (Train)", leave=False)
    
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        r = np.random.rand(1)
        if r < 0.5:
            images, targets_a, targets_b, lam = cutmix_data(images, labels, alpha=1.0)
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = cutmix_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = criterion(outputs, labels)
        # ------------------------------------

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        # correct_predictions += (predicted == labels).sum().item() 
        # total_predictions += labels.size(0)
        
        train_pbar.set_postfix(loss=loss.item())

    # train_acc = correct_predictions / total_predictions if total_predictions > 0 else 0

    finetune_model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / len(val_dataset)
    
    scheduler.step()

    epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(finetune_model.state_dict())
        torch.save(best_model_wts, 'best_finetuned_model_cutmix.pth')
        epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f}")

print()
print(f"Finished fine-tuning. Best Validation Accuracy: {best_val_acc:.4f}")

Loading pre-trained backbone weights...
Starting supervised fine-tuning with CutMix...


Fine-tuning Progress:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 1 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 2 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 3 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

...

Epoch 397 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 398 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 399 (Train):   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 400 (Train):   0%|          | 0/15 [00:00<?, ?it/s]


Finished fine-tuning. Best Validation Accuracy: 0.7356


In [9]:
print("\n--- Starting Stage 2: Fine-tuning the entire model with CutMix ---")
# Размораживаем все слои
for param in finetune_model.parameters():
    param.requires_grad = True

epoch_pbar = tqdm(range(FINETUNE_EPOCHS), desc="Full Fine-tuning")

for epoch in epoch_pbar:
    finetune_model.train()
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Применяем CutMix с вероятностью 50%
        if np.random.rand() < 0.5:
            images, targets_a, targets_b, lam = cutmix_data(images, labels, alpha=1.0)
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = cutmix_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        train_pbar.set_postfix(loss=loss.item())

    # Валидация
    finetune_model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / len(val_dataset)
    
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(finetune_model.state_dict(), 'best_model_final_enhanced.pth')
        epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f} (Saved!)")
    else:
        epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f}")

print(f"\nFinished fine-tuning. Best Validation Accuracy: {best_val_acc:.4f}")

# Не забудь обновить имя файла для сабмита на 'best_model_final_enhanced.pth'


--- Starting Stage 2: Fine-tuning the entire model with CutMix ---


Full Fine-tuning:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/15 [00:00<?, ?it/s]

...

Epoch 397:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 398:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 399:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 400:   0%|          | 0/15 [00:00<?, ?it/s]


Finished fine-tuning. Best Validation Accuracy: 0.7596


In [10]:
class TestDataset(Dataset):
    """Dataset for test images."""
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.test_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TestDataset(TEST_PATH, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

final_model = models.resnet18(num_classes=10)
final_model.load_state_dict(torch.load('best_model_final_enhanced.pth'))
final_model = final_model.to(DEVICE)
final_model.eval()

predictions = []
image_ids = []

print("Generating predictions on the test set...")
with torch.no_grad():
    for images, fnames in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, predicted_indices = torch.max(outputs, 1)

        predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])
        image_ids.extend(fnames)

submission_df = pd.DataFrame({
    'id': image_ids,
    'class': predictions
})

submission_df.to_csv('submission_rotations_ffff.csv', index=False)

print("Submission file 'submission_rotations_ffff.csv' created successfully!")
print(submission_df.head())

Generating predictions on the test set...


  0%|          | 0/17 [00:00<?, ?it/s]

Submission file 'submission_rotations_ffff.csv' created successfully!
         id     class
0     0.jpg       dog
1     1.jpg     horse
2    10.jpg     sheep
3   100.jpg    spider
4  1000.jpg  elephant


In [4]:
# %% [markdown]
# ## Улучшенный Fine-tuning: CutMix + Дискриминационные LR + Label Smoothing + AdamW

# %%
# Функции для CutMix (остаются без изменений)
def cutmix_data(x, y, alpha=1.0, device='cuda'):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    y_a, y_b = y, y[index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, y_a, y_b, lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def cutmix_criterion(criterion, outputs, y_a, y_b, lam):
    return lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)

# %%
# Трансформации и данные (остаются без изменений)
finetune_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
full_labeled_dataset = datasets.ImageFolder(LABELED_PATH, transform=finetune_transforms)
train_size = int(0.9 * len(full_labeled_dataset))
val_size = len(full_labeled_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_labeled_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
class_names = full_labeled_dataset.classes

# ==========================================================
# Инициализация модели и ОПТИМИЗАТОРА с дискриминационными LR
# ==========================================================
finetune_model = models.resnet18(num_classes=10)
print("Loading pre-trained multitask backbone weights...")
backbone_to_load = nn.Sequential(*list(models.resnet18().children())[:-1])
backbone_to_load.load_state_dict(torch.load('resnet18_backbone_multitask_128.pth'))
finetune_model.load_state_dict(backbone_to_load.state_dict(), strict=False)
finetune_model = finetune_model.to(DEVICE)

# --- Создаем группы параметров для дискриминационных LR ---
base_lr = LEARNING_RATE_FINETUNE
param_groups = [
    {'params': list(finetune_model.conv1.parameters()) + list(finetune_model.bn1.parameters()) + \
               list(finetune_model.layer1.parameters()) + list(finetune_model.layer2.parameters()), 'lr': base_lr / 25},
    {'params': list(finetune_model.layer3.parameters()) + list(finetune_model.layer4.parameters()), 'lr': base_lr / 5},
    {'params': finetune_model.fc.parameters(), 'lr': base_lr}
]

# Используем AdamW, Label Smoothing и CosineAnnealingLR
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(param_groups, lr=base_lr, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

best_val_acc = 0.0

# ==========================================================
# Цикл обучения с CutMix и дискриминационными LR
# ==========================================================
print("\n--- Starting fine-tuning with CutMix and Discriminative LR ---")
epoch_pbar = tqdm(range(FINETUNE_EPOCHS), desc="Fine-tuning")

for epoch in epoch_pbar:
    finetune_model.train()
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Применяем CutMix с вероятностью 50%
        if np.random.rand() < 0.5:
            images, targets_a, targets_b, lam = cutmix_data(images, labels, alpha=1.0)
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = cutmix_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = finetune_model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        train_pbar.set_postfix(loss=loss.item(), lr_head=f"{optimizer.param_groups[2]['lr']:.1e}")

    # Валидация
    finetune_model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = finetune_model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / len(val_dataset)
    
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(finetune_model.state_dict(), 'best_model_final_enhanced.pth')
        epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f} (Saved!)")
    else:
        epoch_pbar.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f}")

print(f"\nFinished fine-tuning. Best Validation Accuracy: {best_val_acc:.4f}")

Loading pre-trained multitask backbone weights...

--- Starting fine-tuning with CutMix and Discriminative LR ---


Fine-tuning:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/15 [00:00<?, ?it/s]

...

Epoch 397:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 398:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 399:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 400:   0%|          | 0/15 [00:00<?, ?it/s]


Finished fine-tuning. Best Validation Accuracy: 0.6010


In [5]:
class TestDataset(Dataset):
    """Dataset for test images."""
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.test_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TestDataset(TEST_PATH, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

final_model = models.resnet18(num_classes=10)
final_model.load_state_dict(torch.load('best_model_final_enhanced.pth'))
final_model = final_model.to(DEVICE)
final_model.eval()

predictions = []
image_ids = []

print("Generating predictions on the test set...")
with torch.no_grad():
    for images, fnames in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = final_model(images)
        _, predicted_indices = torch.max(outputs, 1)

        predictions.extend([class_names[i] for i in predicted_indices.cpu().numpy()])
        image_ids.extend(fnames)

submission_df = pd.DataFrame({
    'id': image_ids,
    'class': predictions
})

submission_df.to_csv('submission_rotations.csv', index=False)

print("Submission file 'submission_rotations.csv' created successfully!")
print(submission_df.head())

Generating predictions on the test set...


  0%|          | 0/17 [00:00<?, ?it/s]

Submission file 'submission_rotations.csv' created successfully!
         id     class
0     0.jpg       dog
1     1.jpg     horse
2    10.jpg       dog
3   100.jpg    spider
4  1000.jpg  elephant
